# TP1 Salud Pública — Etapa 2: auditoría del dataset

Ejecutar todas las celdas en orden desde la raíz del repositorio o desde `notebooks/`. Dependencias: pandas, openpyxl e IPython; para ejecución automatizada: nbformat, nbclient e ipykernel. No se modifica el DataFrame cargado ni se exportan datos procesados. Las tablas son diagnósticos de calidad, no EDA del caso. `informe` reúne el texto reproducible del reporte al finalizar.

In [1]:
from pathlib import Path
import sys, hashlib, platform, unicodedata
from itertools import combinations
import pandas as pd
import numpy as np
import openpyxl
from IPython.display import display, Markdown

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dataset_salud_publica_2500.xlsx").exists()
             and (p / "reports/01_objetivos.md").exists()), None)
assert root is not None, "Ejecutar dentro del repositorio."
paths = [root / "dataset_salud_publica_2500.xlsx", root / "reports/01_objetivos.md"]
hashes = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
objetivos = paths[1].read_text(encoding="utf-8")
df = pd.read_excel(paths[0], sheet_name="Salud Publica", engine="openpyxl")
original = df.copy(deep=True)
n = len(df)
parts = []
import re
def format_markdown(text):
    lines = text.splitlines()
    result = []
    i = 0
    while i < len(lines):
        if not lines[i].lstrip().startswith("|"):
            result.append(lines[i].rstrip())
            i += 1
            continue
        block = []
        while i < len(lines) and lines[i].lstrip().startswith("|"):
            block.append(lines[i])
            i += 1
        rows = [[c.strip() for c in re.split(r"(?<!\\)\|", line.strip())[1:-1]] for line in block]
        if len(rows) < 2 or not all(re.fullmatch(r":?-{3,}:?", c) for c in rows[1]):
            result.extend(block)
            continue
        assert all(len(row) == len(rows[0]) for row in rows), "Tabla irregular"
        numeric = [bool(rows[2:]) and all(re.fullmatch(r"-?\d+(?:\.\d+)?%?|NA", row[j]) for row in rows[2:])
                   for j in range(len(rows[0]))]
        widths = [max(3, *(len(row[j]) for k,row in enumerate(rows) if k != 1)) for j in range(len(rows[0]))]
        for k,row in enumerate(rows):
            if k == 1:
                values = ["-"*(w-1)+":" if numeric[j] else "-"*w for j,w in enumerate(widths)]
            else:
                values = [value.rjust(widths[j]) if numeric[j] and k > 1 else value.ljust(widths[j]) for j,value in enumerate(row)]
            result.append("| " + " | ".join(values) + " |")
    return "\n".join(result).rstrip() + "\n"

def emit(text):
    text = format_markdown(text).rstrip()
    parts.append(text)
    display(Markdown(text))
def table(frame, index=False):
    t = frame.reset_index() if index else frame
    def fmt(v):
        if pd.isna(v): return "NA"
        if isinstance(v, (float, np.floating)): return f"{v:.4f}"
        return str(v).replace("|", "\\|").replace("\n", " ")
    rows = ["| " + " | ".join(map(str, t.columns)) + " |",
            "| " + " | ".join(["---"] * len(t.columns)) + " |"]
    rows += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in t.itertuples(index=False, name=None)]
    return "\n".join(rows)
def pct(k): return 100 * k / n
emit("# TP1 Salud Pública — Auditoría del dataset\n\n"
     "Etapa 2 exclusivamente. Fuente: Excel ficticio y reports/01_objetivos.md. "
     "No se aplicaron limpieza, imputación, corrección, eliminación, gráficos, modelos ni análisis de asociaciones. "
     "Los porcentajes usan todas las filas como denominador, salvo indicación expresa. "
     "Los valores nulos son los reconocidos por pandas con los parámetros predeterminados de read_excel.\n\n"
     f"Entorno: Python {platform.python_version()}, pandas {pd.__version__}, NumPy {np.__version__}, openpyxl {openpyxl.__version__}. "
     "Las versiones pueden afectar los tipos inferidos y el consumo de memoria.")


# TP1 Salud Pública — Auditoría del dataset

Etapa 2 exclusivamente. Fuente: Excel ficticio y reports/01_objetivos.md. No se aplicaron limpieza, imputación, corrección, eliminación, gráficos, modelos ni análisis de asociaciones. Los porcentajes usan todas las filas como denominador, salvo indicación expresa. Los valores nulos son los reconocidos por pandas con los parámetros predeterminados de read_excel.

Entorno: Python 3.12.3, pandas 3.0.5, NumPy 2.5.3, openpyxl 3.1.5. Las versiones pueden afectar los tipos inferidos y el consumo de memoria.

In [2]:
expected = ["ID_Paciente", "Region", "Edad", "Genero", "Condicion_Salud", "Cobertura_Salud",
            "Frecuencia_Atencion", "Tiempo_Espera_min", "Acceso_Medicacion", "Satisfaccion"]
assert list(df.columns) == expected
memory = df.memory_usage(index=True, deep=True)
structure = pd.DataFrame({"Columna": df.columns, "Tipo pandas": [str(v) for v in df.dtypes],
                          "Memoria bytes": [memory[c] for c in df.columns]})
emit("## 1. Carga y estructura\n\n"
     f"Hoja: Salud Publica. Dimensiones reales: **{n} filas × {df.shape[1]} columnas**. "
     f"Coincide con los 2500 registros declarados: **{'sí' if n == 2500 else 'no'}**.\n\n"
     + table(structure) + "\n\n"
     f"Memoria profunda total, incluido índice: **{memory.sum()} bytes ({memory.sum()/1024:.2f} KiB)**. "
     f"Índice: {memory['Index']} bytes. Es memoria del DataFrame, no tamaño del archivo.\n\n"
     "Primeras cinco filas, únicamente como control de carga:\n\n" + table(df.head(5)))


## 1. Carga y estructura

Hoja: Salud Publica. Dimensiones reales: **2500 filas × 10 columnas**. Coincide con los 2500 registros declarados: **sí**.

| Columna             | Tipo pandas | Memoria bytes |
| ------------------- | ----------- | ------------: |
| ID_Paciente         | int64       |         20000 |
| Region              | str         |        137613 |
| Edad                | int64       |         20000 |
| Genero              | str         |        127481 |
| Condicion_Salud     | str         |        139998 |
| Cobertura_Salud     | str         |        145106 |
| Frecuencia_Atencion | int64       |         20000 |
| Tiempo_Espera_min   | float64     |         20000 |
| Acceso_Medicacion   | str         |        154620 |
| Satisfaccion        | float64     |         20000 |

Memoria profunda total, incluido índice: **804950 bytes (786.08 KiB)**. Índice: 132 bytes. Es memoria del DataFrame, no tamaño del archivo.

Primeras cinco filas, únicamente como control de carga:

| ID_Paciente | Region       | Edad | Genero | Condicion_Salud | Cobertura_Salud | Frecuencia_Atencion | Tiempo_Espera_min | Acceso_Medicacion | Satisfaccion |
| ----------: | ------------ | ---: | ------ | --------------- | --------------- | ------------------: | ----------------: | ----------------- | -----------: |
|           1 | CABA         |   23 | Otro   | Aguda           | Sin cobertura   |                   3 |          158.0000 | Sí                |       2.0000 |
|           2 | Sur          |   58 | M      | Cronica         | Privada         |                  12 |           20.0000 | Sí                |       4.0000 |
|           3 | Sur          |   49 | F      | Aguda           | Privada         |                   3 |           11.0000 | Sí                |       4.0000 |
|           4 | Buenos Aires |   83 | Otro   | Cronica         | Sin cobertura   |                  16 |          140.0000 | Sí                |       3.0000 |
|           5 | Norte        |   18 | F      | Saludable       | Sin cobertura   |                   0 |          175.0000 | No                |       1.0000 |

In [3]:
ids = df["ID_Paciente"]
id_extra = ids.notna() & ids.duplicated(keep="first")
id_all = ids.notna() & ids.duplicated(keep=False)
full_extra = df.duplicated(keep="first")
full_all = df.duplicated(keep=False)
id_summary = pd.DataFrame([
    ("Valores presentes", ids.notna().sum()), ("Valores únicos no nulos", ids.nunique(dropna=True)),
    ("Repeticiones adicionales de ID", id_extra.sum()), ("Filas en grupos de ID repetido", id_all.sum()),
    ("Valores nulos", ids.isna().sum())], columns=["Control", "Cantidad"])
emit("## 2. Identificadores\n\n" + table(id_summary) +
     "\n\nRepeticiones adicionales excluye la primera aparición; filas involucradas incluye todas las apariciones. "
     "Los nulos se cuentan por separado. El ID se trata como identificador nominal.")
if id_all.any():
    emit("Ejemplos de ID repetidos (máximo cinco filas):\n\n" + table(df.loc[id_all].head(5)))


## 2. Identificadores

| Control                        | Cantidad |
| ------------------------------ | -------: |
| Valores presentes              |     2500 |
| Valores únicos no nulos        |     2500 |
| Repeticiones adicionales de ID |        0 |
| Filas en grupos de ID repetido |        0 |
| Valores nulos                  |        0 |

Repeticiones adicionales excluye la primera aparición; filas involucradas incluye todas las apariciones. Los nulos se cuentan por separado. El ID se trata como identificador nominal.

In [4]:
missing = df.isna()
missing_counts = missing.sum()
missing_table = pd.DataFrame({"Columna": df.columns, "Nulos": missing_counts.to_numpy(),
                               "Porcentaje": missing_counts.to_numpy()/n*100})
row_missing = missing.sum(axis=1)
missing_cols = list(missing_counts[missing_counts > 0].index)
patterns = missing.apply(lambda row: ", ".join(row.index[row]) or "Sin faltantes", axis=1).value_counts()
pattern_table = pd.DataFrame({"Columnas faltantes en la misma fila": patterns.index,
                              "Filas": patterns.values, "Porcentaje": patterns.values/n*100})
pair_rows = [(a,b,int((missing[a]&missing[b]).sum())) for a,b in combinations(missing_cols,2)]
emit("## 3. Valores faltantes\n\n" + table(missing_table) +
     "\n\nColumnas con faltantes: " + (", ".join(missing_cols) or "ninguna") + ".\n\n"
     f"Filas con al menos un faltante: **{(row_missing>0).sum()} ({pct((row_missing>0).sum()):.2f}%)**. "
     f"Con más de uno: **{(row_missing>1).sum()} ({pct((row_missing>1).sum()):.2f}%)**.\n\n"
     "Patrones exactos de coincidencia (incluye filas completas):\n\n" + table(pattern_table))
if pair_rows:
    emit("Coincidencias por par de columnas:\n\n" + table(pd.DataFrame(pair_rows,columns=["Columna A","Columna B","Filas con ambos nulos"])) +
         "\n\nEstos conteos describen coincidencias; no identifican el mecanismo ni la causa de ausencia.")


## 3. Valores faltantes

| Columna             | Nulos | Porcentaje |
| ------------------- | ----: | ---------: |
| ID_Paciente         |     0 |     0.0000 |
| Region              |     0 |     0.0000 |
| Edad                |     0 |     0.0000 |
| Genero              |     0 |     0.0000 |
| Condicion_Salud     |     0 |     0.0000 |
| Cobertura_Salud     |     0 |     0.0000 |
| Frecuencia_Atencion |     0 |     0.0000 |
| Tiempo_Espera_min   |    74 |     2.9600 |
| Acceso_Medicacion   |     0 |     0.0000 |
| Satisfaccion        |    72 |     2.8800 |

Columnas con faltantes: Tiempo_Espera_min, Satisfaccion.

Filas con al menos un faltante: **143 (5.72%)**. Con más de uno: **3 (0.12%)**.

Patrones exactos de coincidencia (incluye filas completas):

| Columnas faltantes en la misma fila | Filas | Porcentaje |
| ----------------------------------- | ----: | ---------: |
| Sin faltantes                       |  2357 |    94.2800 |
| Tiempo_Espera_min                   |    71 |     2.8400 |
| Satisfaccion                        |    69 |     2.7600 |
| Tiempo_Espera_min, Satisfaccion     |     3 |     0.1200 |

Coincidencias por par de columnas:

| Columna A         | Columna B    | Filas con ambos nulos |
| ----------------- | ------------ | --------------------: |
| Tiempo_Espera_min | Satisfaccion |                     3 |

Estos conteos describen coincidencias; no identifican el mecanismo ni la causa de ausencia.

In [5]:
emit("## 4. Duplicados\n\n" + table(pd.DataFrame([
    ("Filas completamente duplicadas, adicionales", int(full_extra.sum())),
    ("Filas involucradas en duplicación completa", int(full_all.sum())),
    ("IDs repetidos, apariciones adicionales", int(id_extra.sum())),
    ("Filas involucradas en IDs repetidos", int(id_all.sum()))
], columns=["Control","Cantidad"])) + "\n\nLa duplicación completa compara las diez columnas, incluido ID_Paciente. No se eliminaron registros.")
if full_all.any():
    emit("Ejemplos de duplicación completa (máximo cinco filas):\n\n"+table(df.loc[full_all].head(5)))


## 4. Duplicados

| Control                                     | Cantidad |
| ------------------------------------------- | -------: |
| Filas completamente duplicadas, adicionales |        0 |
| Filas involucradas en duplicación completa  |        0 |
| IDs repetidos, apariciones adicionales      |        0 |
| Filas involucradas en IDs repetidos         |        0 |

La duplicación completa compara las diez columnas, incluido ID_Paciente. No se eliminaron registros.

In [6]:
categorical = ["Region","Genero","Condicion_Salud","Cobertura_Salud","Acceso_Medicacion","Satisfaccion"]
domains = {"Region": ["Norte","Centro","Sur","CABA","Buenos Aires"],
           "Condicion_Salud": ["Saludable","Crónica","Aguda"],
           "Cobertura_Salud": ["Pública","Privada","Sin cobertura"],
           "Acceso_Medicacion": ["Sí","No"], "Satisfaccion": [1,2,3,4,5]}
outside = {}
format_rows = []
def key(text):
    return "".join(c for c in unicodedata.normalize("NFD", text.strip().casefold()) if not unicodedata.combining(c))
emit("## 5. Variables categóricas y escala ordinal\n\n"
     "Las tablas enumeran todos los valores únicos, incluyendo NA si existe; porcentajes sobre todas las filas. "
     "repr conserva visibles los espacios en los textos. Satisfaccion es conceptualmente ordinal 1–5, aunque pandas infiera un tipo numérico. "
     "Genero no tiene catálogo documentado: sus categorías observadas no pueden declararse fuera de dominio sin una definición adicional. "
     "Los dominios de las demás columnas se toman de la Etapa 1.")
for c in categorical:
    counts = df[c].value_counts(dropna=False, sort=False)
    freq = pd.DataFrame({"Valor literal": [repr(v) if pd.notna(v) else "NA" for v in counts.index],
                         "Frecuencia": counts.values, "Porcentaje": counts.values/n*100})
    emit("### " + c + "\n\n" + table(freq))
    vals = [v for v in df[c].dropna().unique() if isinstance(v,str)]
    groups = {}
    for v in vals: groups.setdefault(key(v),[]).append(v)
    collisions = [vs for vs in groups.values() if len(vs)>1]
    spaces = df[c].map(lambda v: isinstance(v,str) and (v != v.strip() or "  " in v or any(ch.isspace() and ch != " " for ch in v)))
    format_rows.append((c,int(spaces.sum()),repr(collisions)))
    if c in domains:
        outside[c] = df[c].notna() & ~df[c].isin(domains[c])
emit("### Escritura y dominios\n\n" +
     table(pd.DataFrame(format_rows,columns=["Columna","Filas con espacios anómalos","Variantes por espacios/caja/tildes"])) +
     "\n\nComparar claves sin espacios exteriores, mayúsculas ni tildes se usa solo para diagnóstico; no se asigna al dataset. "
     "La comparación exacta con el dominio también detecta variantes únicas de escritura.\n\n" +
     table(pd.DataFrame([(c,int(m.sum()),pct(m.sum()),repr(df.loc[m,c].unique().tolist())) for c,m in outside.items()],
                        columns=["Columna","Fuera del dominio literal","Porcentaje","Valores"])))


## 5. Variables categóricas y escala ordinal

Las tablas enumeran todos los valores únicos, incluyendo NA si existe; porcentajes sobre todas las filas. repr conserva visibles los espacios en los textos. Satisfaccion es conceptualmente ordinal 1–5, aunque pandas infiera un tipo numérico. Genero no tiene catálogo documentado: sus categorías observadas no pueden declararse fuera de dominio sin una definición adicional. Los dominios de las demás columnas se toman de la Etapa 1.

### Region

| Valor literal  | Frecuencia | Porcentaje |
| -------------- | ---------: | ---------: |
| 'CABA'         |        498 |    19.9200 |
| 'Sur'          |        505 |    20.2000 |
| 'Buenos Aires' |        518 |    20.7200 |
| 'Norte'        |        484 |    19.3600 |
| 'Centro'       |        495 |    19.8000 |

### Genero

| Valor literal | Frecuencia | Porcentaje |
| ------------- | ---------: | ---------: |
| 'Otro'        |        827 |    33.0800 |
| 'M'           |        833 |    33.3200 |
| 'F'           |        840 |    33.6000 |

### Condicion_Salud

| Valor literal | Frecuencia | Porcentaje |
| ------------- | ---------: | ---------: |
| 'Aguda'       |        829 |    33.1600 |
| 'Cronica'     |        843 |    33.7200 |
| 'Saludable'   |        828 |    33.1200 |

### Cobertura_Salud

| Valor literal   | Frecuencia | Porcentaje |
| --------------- | ---------: | ---------: |
| 'Sin cobertura' |        851 |    34.0400 |
| 'Privada'       |        843 |    33.7200 |
| 'Publica'       |        806 |    32.2400 |

### Acceso_Medicacion

| Valor literal | Frecuencia | Porcentaje |
| ------------- | ---------: | ---------: |
| 'Sí'          |       1695 |    67.8000 |
| 'No'          |        805 |    32.2000 |

### Satisfaccion

| Valor literal | Frecuencia | Porcentaje |
| ------------: | ---------: | ---------: |
|           2.0 |        328 |    13.1200 |
|           4.0 |        794 |    31.7600 |
|           3.0 |        576 |    23.0400 |
|           1.0 |        139 |     5.5600 |
|           5.0 |        591 |    23.6400 |
|            NA |         72 |     2.8800 |

### Escritura y dominios

| Columna           | Filas con espacios anómalos | Variantes por espacios/caja/tildes |
| ----------------- | --------------------------: | ---------------------------------- |
| Region            |                           0 | []                                 |
| Genero            |                           0 | []                                 |
| Condicion_Salud   |                           0 | []                                 |
| Cobertura_Salud   |                           0 | []                                 |
| Acceso_Medicacion |                           0 | []                                 |
| Satisfaccion      |                           0 | []                                 |

Comparar claves sin espacios exteriores, mayúsculas ni tildes se usa solo para diagnóstico; no se asigna al dataset. La comparación exacta con el dominio también detecta variantes únicas de escritura.

| Columna           | Fuera del dominio literal | Porcentaje | Valores     |
| ----------------- | ------------------------: | ---------: | ----------- |
| Region            |                         0 |     0.0000 | []          |
| Condicion_Salud   |                       843 |    33.7200 | ['Cronica'] |
| Cobertura_Salud   |                       806 |    32.2400 | ['Publica'] |
| Acceso_Medicacion |                         0 |     0.0000 | []          |
| Satisfaccion      |                         0 |     0.0000 | []          |

In [7]:
numeric = ["Edad","Frecuencia_Atencion","Tiempo_Espera_min"]
stat_rows, iqr_rows, iqr_masks = [], [], {}
for c in numeric:
    s = df[c]
    q1, q3 = s.quantile([.25,.75])
    iqr = q3-q1
    lo, hi = q1-1.5*iqr, q3+1.5*iqr
    mask = s.notna() & ((s<lo)|(s>hi))
    iqr_masks[c] = mask
    stat_rows.append([c,s.count(),s.mean(),s.median(),s.std(ddof=1),s.min(),q1,q3,s.max(),iqr])
    iqr_rows.append([c,lo,hi,int(mask.sum()),pct(mask.sum())])
emit("## 6. Variables numéricas\n\n"
     "Estadísticas calculadas sobre valores presentes, sin imputar. Desviación estándar muestral (ddof=1); "
     "cuartiles con interpolación lineal predeterminada de pandas. El porcentaje de alertas usa las 2500 filas.\n\n" +
     table(pd.DataFrame(stat_rows,columns=["Variable","count","Media","Mediana","Desv. estándar","Mínimo","Q1","Q3","Máximo","IQR"])) +
     "\n\nRegla diagnóstica: valores estrictamente menores que Q1 − 1.5 × IQR o mayores que Q3 + 1.5 × IQR. "
     "Los límites son estadísticos, no límites de validez del dominio.\n\n" +
     table(pd.DataFrame(iqr_rows,columns=["Variable","Límite inferior","Límite superior","Alertas IQR","Porcentaje"])))
for c,m in iqr_masks.items():
    if m.any():
        emit("Ejemplos IQR de " + c + " (máximo cinco):\n\n" + table(df.loc[m,["ID_Paciente",c]].head(5)))
emit("### Interpretación de las alertas numéricas\n\n"
     "- **Imposible:** edad o duración negativas, consultas negativas o fraccionarias y valores no finitos contradicen sus conceptos. No se encontraron.\n"
     "- **Sospechoso:** un valor requiere verificación contextual o documental; la regla IQR no basta para llamarlo error. No aparecieron alertas IQR ni otras infracciones numéricas objetivas en estos controles.\n"
     "- **Extremo pero plausible:** los máximos observados de 90 en Edad, 20 consultas anuales y 180 minutos de espera no contradicen un límite documentado y no activan IQR. Se consideran extremos del archivo potencialmente plausibles, no errores; no se certifica su exactitud.\n\n"
     "El caso no fija un máximo válido de edad ni de espera. No se inventa un umbral superior; "
     "Edad=0 no es automáticamente imposible. La interpretación de edad en años no está confirmada por el documento.")


## 6. Variables numéricas

Estadísticas calculadas sobre valores presentes, sin imputar. Desviación estándar muestral (ddof=1); cuartiles con interpolación lineal predeterminada de pandas. El porcentaje de alertas usa las 2500 filas.

| Variable            | count | Media   | Mediana | Desv. estándar | Mínimo | Q1      | Q3      | Máximo   | IQR     |
| ------------------- | ----: | ------: | ------: | -------------: | -----: | ------: | ------: | -------: | ------: |
| Edad                |  2500 | 45.3996 | 45.0000 |        26.4906 | 0.0000 | 22.0000 | 69.0000 |  90.0000 | 47.0000 |
| Frecuencia_Atencion |  2500 |  6.1884 |  4.0000 |         5.5810 | 0.0000 |  2.0000 | 10.0000 |  20.0000 |  8.0000 |
| Tiempo_Espera_min   |  2426 | 64.5808 | 54.0000 |        47.7098 | 5.0000 | 23.0000 | 99.0000 | 180.0000 | 76.0000 |

Regla diagnóstica: valores estrictamente menores que Q1 − 1.5 × IQR o mayores que Q3 + 1.5 × IQR. Los límites son estadísticos, no límites de validez del dominio.

| Variable            | Límite inferior | Límite superior | Alertas IQR | Porcentaje |
| ------------------- | --------------: | --------------: | ----------: | ---------: |
| Edad                |        -48.5000 |        139.5000 |           0 |     0.0000 |
| Frecuencia_Atencion |        -10.0000 |         22.0000 |           0 |     0.0000 |
| Tiempo_Espera_min   |        -91.0000 |        213.0000 |           0 |     0.0000 |

### Interpretación de las alertas numéricas

- **Imposible:** edad o duración negativas, consultas negativas o fraccionarias y valores no finitos contradicen sus conceptos. No se encontraron.
- **Sospechoso:** un valor requiere verificación contextual o documental; la regla IQR no basta para llamarlo error. No aparecieron alertas IQR ni otras infracciones numéricas objetivas en estos controles.
- **Extremo pero plausible:** los máximos observados de 90 en Edad, 20 consultas anuales y 180 minutos de espera no contradicen un límite documentado y no activan IQR. Se consideran extremos del archivo potencialmente plausibles, no errores; no se certifica su exactitud.

El caso no fija un máximo válido de edad ni de espera. No se inventa un umbral superior; Edad=0 no es automáticamente imposible. La interpretación de edad en años no está confirmada por el documento.

In [8]:
checks = {}
for c in numeric:
    checks[c + ": negativos"] = df[c].notna() & (df[c]<0)
    checks[c + ": no finitos"] = df[c].notna() & ~np.isfinite(df[c])
checks["Frecuencia_Atencion: no entera"] = df.Frecuencia_Atencion.notna() & (df.Frecuencia_Atencion % 1 != 0)
checks["Satisfaccion: fuera de 1–5 entero"] = outside["Satisfaccion"]
checks["Acceso_Medicacion: distinto de Sí/No"] = outside["Acceso_Medicacion"]
emit("## 7. Consistencia lógica\n\n" +
     table(pd.DataFrame([(label,int(m.sum()),pct(m.sum())) for label,m in checks.items()],
                        columns=["Control","Filas afectadas","Porcentaje"])) +
     "\n\nNo se detectaron infracciones en estos controles. Los faltantes se informan por separado y no se consideran valores fuera de dominio. "
     "No hay una restricción determinista documentada que obligue a una combinación específica entre condición, consultas, cobertura, acceso, espera y satisfacción. "
     "Por ello no se interpreta una combinación inusual como inconsistencia. En particular, cero consultas anuales junto con espera o satisfacción "
     "no prueba un error sin conocer los períodos de referencia. No se contrastaron las asociaciones incorporadas al dataset.")


## 7. Consistencia lógica

| Control                              | Filas afectadas | Porcentaje |
| ------------------------------------ | --------------: | ---------: |
| Edad: negativos                      |               0 |     0.0000 |
| Edad: no finitos                     |               0 |     0.0000 |
| Frecuencia_Atencion: negativos       |               0 |     0.0000 |
| Frecuencia_Atencion: no finitos      |               0 |     0.0000 |
| Tiempo_Espera_min: negativos         |               0 |     0.0000 |
| Tiempo_Espera_min: no finitos        |               0 |     0.0000 |
| Frecuencia_Atencion: no entera       |               0 |     0.0000 |
| Satisfaccion: fuera de 1–5 entero    |               0 |     0.0000 |
| Acceso_Medicacion: distinto de Sí/No |               0 |     0.0000 |

No se detectaron infracciones en estos controles. Los faltantes se informan por separado y no se consideran valores fuera de dominio. No hay una restricción determinista documentada que obligue a una combinación específica entre condición, consultas, cobertura, acceso, espera y satisfacción. Por ello no se interpreta una combinación inusual como inconsistencia. En particular, cero consultas anuales junto con espera o satisfacción no prueba un error sin conocer los períodos de referencia. No se contrastaron las asociaciones incorporadas al dataset.

In [9]:
# Clasificación de incidencias observadas; ningún tratamiento se aplica.
issues = []
for c in missing_cols:
    k = int(missing_counts[c])
    issues.append(["IMPORTANTE",c,"Valores faltantes",k,pct(k),
                   "Reduce casos disponibles; el tratamiento puede cambiar denominadores y resultados.",
                   "Investigar origen; conservar NA y usar casos disponibles por análisis; evaluar exclusión específica o imputación justificada en otra etapa."])
for c,m in outside.items():
    if m.any():
        vals = df.loc[m,c].unique().tolist()
        known_keys = {key(v) for v in domains[c] if isinstance(v,str)}
        spelling_only = all(isinstance(v,str) and key(v) in known_keys for v in vals)
        issues.append(["MENOR" if spelling_only else "IMPORTANTE",c,
                       "Diferencia literal respecto del documento: " + repr(vals),int(m.sum()),pct(m.sum()),
                       "Filtros o cruces con el catálogo acentuado pueden omitir estos registros." if spelling_only else "Dominio no válido para análisis directo.",
                       "Documentar correspondencia y conservar etiquetas; o acordar normalización posterior." if spelling_only else "Revisar fuente y decidir tratamiento posterior."])
quality = pd.DataFrame(issues,columns=["Severidad","Variable","Problema","Registros","Porcentaje","Posible impacto","Alternativas, no aplicadas"])
critical = sum(row[0] == "CRÍTICO" for row in issues)
emit("## 8. Reporte de calidad\n\n"
     "Criterios: CRÍTICO impediría o invalidaría análisis posteriores; IMPORTANTE requiere una decisión previa; "
     "MENOR conviene documentarlo pero probablemente no altera el análisis; SIN PROBLEMAS DETECTADOS se limita a los controles realizados.\n\n" +
     table(quality) + "\n\n"
     f"**Problemas críticos detectados: {critical}.** "
     "Se registran dos incidencias IMPORTANTES por faltantes y dos MENORES de escritura. "
     "Las incidencias pueden afectar las mismas filas: no deben sumarse como pacientes distintos.\n\n"
     "La discrepancia Cronica/Crónica afecta 843 filas (33.72%); Publica/Pública afecta 806 (32.24%). "
     "Son diferencias ortográficas frente al catálogo documental, no categorías semánticamente nuevas ni mezclas de etiquetas dentro de la columna. "
     "No hay valores distintos de Sí/No en acceso ni valores presentes fuera de 1–5 en satisfacción.\n\n"
     "**SIN PROBLEMAS DETECTADOS:** dimensiones y encabezados esperados; integridad y unicidad del ID; duplicación completa; "
     "dominio de Region, Acceso_Medicacion y Satisfaccion; espacios anómalos y variantes internas de escritura en las categorías inspeccionadas; "
     "negativos, no finitos, consultas fraccionarias y alertas IQR. Cada uno afecta 0 registros (0%). "
     "Genero tiene F, M y Otro; al no existir catálogo documental, su validez externa no está verificada. "
     "No se propone recodificación por ese solo motivo.")
emit("## 9. Decisiones que debe tomar el equipo antes de limpiar\n\n"
     "1. **Faltantes en Tiempo_Espera_min (74; 2.96%) y Satisfaccion (72; 2.88%):** "
     "decidir si se puede aclarar su origen y cómo se manejarán en cada análisis. "
     "Hay 143 filas con al menos uno (5.72%) y 3 con ambos (0.12%); eliminar toda fila incompleta excluiría 143, "
     "aunque 71 carecen solo de espera y 69 solo de satisfacción. Considerar mantener NA con denominadores explícitos, "
     "exclusión limitada a variables necesarias o una imputación fundamentada posteriormente. No hay evidencia aquí para elegir un mecanismo de ausencia o un método de imputación.\n"
     "2. **Diferencias de tildes frente al documento:** acordar documentar Cronica ↔ Crónica y Publica ↔ Pública manteniendo las etiquetas originales, "
     "o normalizarlas en una futura copia de trabajo si se necesita compatibilidad con un catálogo. La normalización no es necesaria por sí sola para calcular resultados.\n\n"
     "No se justifican eliminación de duplicados, recorte de extremos ni otras correcciones con los resultados de esta auditoría. "
     "No se creó una variable de vulnerabilidad ni un dataset procesado. La auditoría finaliza aquí.")


## 8. Reporte de calidad

Criterios: CRÍTICO impediría o invalidaría análisis posteriores; IMPORTANTE requiere una decisión previa; MENOR conviene documentarlo pero probablemente no altera el análisis; SIN PROBLEMAS DETECTADOS se limita a los controles realizados.

| Severidad  | Variable          | Problema                                               | Registros | Porcentaje | Posible impacto                                                                    | Alternativas, no aplicadas                                                                                                                  |
| ---------- | ----------------- | ------------------------------------------------------ | --------: | ---------: | ---------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- |
| IMPORTANTE | Tiempo_Espera_min | Valores faltantes                                      |        74 |     2.9600 | Reduce casos disponibles; el tratamiento puede cambiar denominadores y resultados. | Investigar origen; conservar NA y usar casos disponibles por análisis; evaluar exclusión específica o imputación justificada en otra etapa. |
| IMPORTANTE | Satisfaccion      | Valores faltantes                                      |        72 |     2.8800 | Reduce casos disponibles; el tratamiento puede cambiar denominadores y resultados. | Investigar origen; conservar NA y usar casos disponibles por análisis; evaluar exclusión específica o imputación justificada en otra etapa. |
| MENOR      | Condicion_Salud   | Diferencia literal respecto del documento: ['Cronica'] |       843 |    33.7200 | Filtros o cruces con el catálogo acentuado pueden omitir estos registros.          | Documentar correspondencia y conservar etiquetas; o acordar normalización posterior.                                                        |
| MENOR      | Cobertura_Salud   | Diferencia literal respecto del documento: ['Publica'] |       806 |    32.2400 | Filtros o cruces con el catálogo acentuado pueden omitir estos registros.          | Documentar correspondencia y conservar etiquetas; o acordar normalización posterior.                                                        |

**Problemas críticos detectados: 0.** Se registran dos incidencias IMPORTANTES por faltantes y dos MENORES de escritura. Las incidencias pueden afectar las mismas filas: no deben sumarse como pacientes distintos.

La discrepancia Cronica/Crónica afecta 843 filas (33.72%); Publica/Pública afecta 806 (32.24%). Son diferencias ortográficas frente al catálogo documental, no categorías semánticamente nuevas ni mezclas de etiquetas dentro de la columna. No hay valores distintos de Sí/No en acceso ni valores presentes fuera de 1–5 en satisfacción.

**SIN PROBLEMAS DETECTADOS:** dimensiones y encabezados esperados; integridad y unicidad del ID; duplicación completa; dominio de Region, Acceso_Medicacion y Satisfaccion; espacios anómalos y variantes internas de escritura en las categorías inspeccionadas; negativos, no finitos, consultas fraccionarias y alertas IQR. Cada uno afecta 0 registros (0%). Genero tiene F, M y Otro; al no existir catálogo documental, su validez externa no está verificada. No se propone recodificación por ese solo motivo.

## 9. Decisiones que debe tomar el equipo antes de limpiar

1. **Faltantes en Tiempo_Espera_min (74; 2.96%) y Satisfaccion (72; 2.88%):** decidir si se puede aclarar su origen y cómo se manejarán en cada análisis. Hay 143 filas con al menos uno (5.72%) y 3 con ambos (0.12%); eliminar toda fila incompleta excluiría 143, aunque 71 carecen solo de espera y 69 solo de satisfacción. Considerar mantener NA con denominadores explícitos, exclusión limitada a variables necesarias o una imputación fundamentada posteriormente. No hay evidencia aquí para elegir un mecanismo de ausencia o un método de imputación.
2. **Diferencias de tildes frente al documento:** acordar documentar Cronica ↔ Crónica y Publica ↔ Pública manteniendo las etiquetas originales, o normalizarlas en una futura copia de trabajo si se necesita compatibilidad con un catálogo. La normalización no es necesaria por sí sola para calcular resultados.

No se justifican eliminación de duplicados, recorte de extremos ni otras correcciones con los resultados de esta auditoría. No se creó una variable de vulnerabilidad ni un dataset procesado. La auditoría finaliza aquí.

In [10]:
pd.testing.assert_frame_equal(df, original, check_exact=True)
for p, digest in hashes.items():
    assert hashlib.sha256(p.read_bytes()).hexdigest() == digest, f"Se modificó {p.name}"
emit("## 10. Reproducibilidad e integridad\n\n"
     "Todos los cálculos y tablas se reproducen ejecutando notebooks/02_auditoria.ipynb en orden. "
     "El notebook reúne este reporte en la variable informe sin escribir sobre las fuentes. "
     "Se verificó igualdad exacta del DataFrame con la copia de carga y SHA-256 de los dos archivos protegidos antes/después.\n\n" +
     table(pd.DataFrame([(p.name,d) for p,d in hashes.items()],columns=["Fuente protegida","SHA-256"])))
informe = "\n\n".join(parts) + "\n"


## 10. Reproducibilidad e integridad

Todos los cálculos y tablas se reproducen ejecutando notebooks/02_auditoria.ipynb en orden. El notebook reúne este reporte en la variable informe sin escribir sobre las fuentes. Se verificó igualdad exacta del DataFrame con la copia de carga y SHA-256 de los dos archivos protegidos antes/después.

| Fuente protegida                | SHA-256                                                          |
| ------------------------------- | ---------------------------------------------------------------- |
| dataset_salud_publica_2500.xlsx | 06449a4ada23692fe43b173198d4d46e05c28a29b5fe796e72eb6926bd1f5100 |
| 01_objetivos.md                 | 845f2358abe8bfd478dd8ccdf7ead2a45df38c46fb8332bc664ed0b2853663d5 |